# Player Acquisition Model
## Predict how much to spend on a player

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn import metrics

from skl2onnx import to_onnx
import onnxruntime as rt
import numpy as np



## Load data

In [ ]:
import pandas as pd
dataset=pd.read_csv("player_training_data_full.csv")

In [ ]:
#x will be training data and y will be the target variable
X = dataset.drop('winning_bid_dollars', axis = 1)
y = dataset['winning_bid_dollars']

In [ ]:
# Training set: X_train, y_train
# Test set: X_test, y_test
# Use 80% for training, save out 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

In [ ]:
# Create the regression model object
model_50th_percentile = GradientBoostingRegressor(loss="quantile", alpha=0.5)

In [ ]:
#Fit your model on the training data
model_50th_percentile.fit(X_train, y_train)

In [ ]:
#Use model to predict values in your test data
y_pred_50 = model_50th_percentile.predict(X_test)

In [ ]:
#Compare the actual values to your predictions
#To evaluation how well your model matches your test data
# Evaluating the Algorithm

print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred_50))  
print('Root Mean Squared Error:', metrics.root_mean_squared_error(y_test, y_pred_50))  

In [ ]:
#90th percentile
#create model object
model_90th_percentile = GradientBoostingRegressor(loss="quantile", alpha=0.9)
#Fit your model on the training data
model_90th_percentile.fit(X_train, y_train)
#Make predictions
y_pred_90 = model_90th_percentile.predict(X_test)

In [ ]:
#10th percentile
#create model object
model_10th_percentile = GradientBoostingRegressor(loss="quantile", alpha=0.1)
#Fit your model on the training data
model_10th_percentile.fit(X_train, y_train)
#Make predictions
y_pred_10 = model_10th_percentile.predict(X_test)

In [ ]:
# Display the predictions for each model
print("50th Percentile Predictions (Median):", y_pred_50)
print("90th Percentile Predictions (Upper Bound):", y_pred_90)
print("10th Percentile Predictions (Lower Bound):", y_pred_10)

## Save these models to ONN

In [ ]:
#Stack inputs into an array
X_stacked = np.column_stack((X['waiver_value_tier'], X['fantasy_regular_season_weeks_remaining'], X['league_budget_pct_remaining']))


# Convert into ONNX format.
# Infer input types from training data
onx = to_onnx(model_10th_percentile, X_stacked[:1])
with open("acquisition_model_10.onnx", "wb") as f:
    f.write(onx.SerializeToString())

# Convert into ONNX format.
# Infer input types from training data
onx = to_onnx(model_50th_percentile, X_stacked[:1])
with open("acquisition_model_50.onnx", "wb") as f:
    f.write(onx.SerializeToString())    

# Convert into ONNX format.
# Infer input types from training data
onx = to_onnx(model_90th_percentile, X_stacked[:1])
with open("acquisition_model_90.onnx", "wb") as f:
    f.write(onx.SerializeToString())